# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# First, list all record sets and their @ids
print("Available Record Sets (by @id and name):")
record_set_ids = []
for rs in metadata.record_sets:
    rs_id = rs['@id']
    name = rs.get('name', '[no name]')
    print(f"- {rs_id}", end='')
    if name:
        print(f" (name: {name})")
    else:
        print()
    record_set_ids.append(rs_id)
if not record_set_ids:
    print("No record sets found in metadata. This dataset's Croissant schema might use only one main table/file.")

# For each record set, list its fields (columns)
rs_fields = {}
for rs in metadata.record_sets:
    print(f"\nFields for record set {rs['@id']}:")
    if hasattr(rs, 'fields') and rs.fields:
        fields = rs.fields
    elif 'fields' in rs:
        fields = rs['fields']
    else:
        fields = []
    rs_fields[rs['@id']] = [fld['@id'] for fld in fields]
    for fld in fields:
        print(f"- {fld['@id']}: {fld.get('name', '[no name]')} ({fld.get('dataType', '[no type]')})")
if not record_set_ids:
    # If no record sets present, try using 'distributions' as source (fallback)
    distributions = getattr(metadata, 'distributions', getattr(metadata, 'distribution', []))
    print("\nDistributions (files) in dataset:")
    for dist in distributions:
        print(json.dumps(dist, indent=2))

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# The record set @id(s) discovered in previous step (use the first, or define explicitly):
if record_set_ids:
    record_sets = record_set_ids
else:
    # Fallback: If no explicit record sets found, try using a standard one (common for most tables)
    record_sets = ['cr:RecordSet']

dataframes = {}
for record_set in record_sets:
    try:
        print(f"Extracting records for record_set {record_set}")
        records = list(dataset.records(record_set=record_set))
        if records:
            dataframes[record_set] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records into DataFrame for {record_set}")
        else:
            print(f"No records found for record set {record_set}")
    except Exception as e:
        print(f"Could not load record set {record_set}: {e}")

# If no dataframes loaded, try the default 'cr:RecordSet'
if not dataframes and 'cr:RecordSet' not in record_sets:
    try:
        print("Trying fallback record set 'cr:RecordSet'")
        records = list(dataset.records(record_set='cr:RecordSet'))
        if records:
            dataframes['cr:RecordSet'] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records into DataFrame for cr:RecordSet")
        else:
            print("No records found for 'cr:RecordSet'")
    except Exception as e:
        print(f"Could not load record set 'cr:RecordSet': {e}")

if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns for DataFrame loaded from record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No DataFrames could be loaded from this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Select a numeric field for analysis
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Choose a likely numeric column (example: protein coverage, MW, or counts)
    numeric_field_names = [
        col for col in df.columns if df[col].dtype in ['int64', 'float64', 'int32', 'float32']
    ]
    # Fall back to scanning columns for numerics (some fields may be object type with numbers)
    if not numeric_field_names:
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notna().sum() > 0:
                    numeric_field_names.append(col)
            except:
                continue
    if numeric_field_names:
        numeric_field = numeric_field_names[0]
        print(f"Using numeric field '{numeric_field}' for filtering and normalization.")

        threshold = df[numeric_field].dropna().mean()
        # Filter by those above the (mean-based) threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize field (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field (choose a non-numeric one)
        group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouping by {group_field} (if it has few unique values)")
            if df[group_field].nunique() < 20:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(f"Grouped mean of {numeric_field} by {group_field}:")
                display(grouped_df.head())
            else:
                print(f"{group_field} has too many unique values, skipping groupby.")
        else:
            print("No non-numeric field available for grouping.")
    else:
        print("No numeric fields found in the DataFrame.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution if available
if dataframes and numeric_field_names:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter with normalized value if that makes sense
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(6, 4))
        plt.scatter(filtered_df[numeric_field], filtered_df[f"{numeric_field}_normalized"])
        plt.title(f"{numeric_field} vs Normalized {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel(f"{numeric_field}_normalized")
        plt.show()

    # Boxplot grouped by group_field if appropriate
    if group_fields:
        group_field = group_fields[0]
        if df[group_field].nunique() < 20:
            plt.figure(figsize=(10, 6))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=45)
            plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!--
This notebook showed how to load, explore, and perform basic EDA on the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

**Key observations:**
- The dataset provides protein-level features by mass spectrometry for extracellular vesicles from human mast cells.
- Data can be filtered and analyzed by protein property fields (e.g., abundance, MW, counts) and grouped by annotation types or experimental conditions if available.
- The Croissant schema makes dataset structure and provenance explicit, and record set, field, or column selection is always reliable by `@id`.

Further analyses can focus on downstream ML preprocessing or advanced bioinformatics queries leveraging the well-described FAIR structure.
-->